# nb_03_gold_publish — the product gate and atomic MERGE

Silver → Gold: refuse pathological snapshots (empty Silver, unexpected `dq_status`, CA=0 —
while empty OR/WA alone never blocks), then publish the contract columns with **one atomic
Delta `MERGE` on `chapter_id`** — update matched, insert new, **delete vanished** — so re-runs
converge and retired chapters leave the product with full time-travel history. Appends the
run summary to the `_runs` Delta table and **exits** with it as JSON.

In the pipeline, `run_id` comes from the Bronze exit value and `silver_counts` from the
Silver exit value. Run by hand, `silver_counts` may be left empty — the notebook then derives
what is derivable from the lake (bronze metadata, quarantine partition, Silver statuses).

In [ ]:
storage_account = ""          # required — ADLS Gen2 account name
lake_container = "lake"
run_id = ""                   # required — the run being published (from nb_01 exit value)
silver_counts = ""            # optional JSON from nb_02 exit value (pipeline passes it)

In [ ]:
%run nb_00_config

In [ ]:
started = datetime.now(timezone.utc)
cfg = init_config(storage_account, lake_container)
if not run_id:
    raise PipelineError("Parameter 'run_id' is required (run being published).")

silver = spark.read.format("delta").load(cfg.silver_path)

rows_silver = silver.count()
if rows_silver == 0:
    raise PipelineError("Silver is empty — refusing to publish an empty Gold snapshot.")
bad_status = silver.filter(~F.col("dq_status").isin(DQ_STATUS_OK, DQ_STATUS_WARNING)).count()
if bad_status:
    raise PipelineError(f"{bad_status} row(s) with unexpected dq_status — aborting publish.")
if silver.filter(F.col("state") == "CA").count() == 0:
    raise PipelineError(
        "CA row count is 0 but CA has a non-zero baseline — refusing to publish. "
        "(Empty OR/WA alone is an expected source data fact and does not block.)"
    )

gold_src = silver.select(*GOLD_COLUMNS)
if DeltaTable.isDeltaTable(spark, cfg.gold_path):
    (DeltaTable.forPath(spark, cfg.gold_path).alias("tgt")
     .merge(gold_src.alias("src"), "tgt.chapter_id = src.chapter_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .whenNotMatchedBySourceDelete()   # full-snapshot source: vanished chapters retire
     .execute())
    log.info("Gold MERGE completed (upsert + delete-vanished on chapter_id).")
else:
    gold_src.write.format("delta").save(cfg.gold_path)
    log.info("Gold created at first publish: %s", cfg.gold_path)

gold = spark.read.format("delta").load(cfg.gold_path)
by_state = {r["state"]: r["n"] for r in gold.groupBy("state").agg(F.count("*").alias("n")).collect()}
gold_counts = {
    "rows_gold": gold.count(),
    "rows_gold_ca": by_state.get("CA", 0),
    "rows_gold_or": by_state.get("OR", 0),
    "rows_gold_wa": by_state.get("WA", 0),
}

# ---- Run summary: prefer the authoritative Silver ledger; derive when run by hand ----
if silver_counts:
    ledger = json.loads(silver_counts)
else:
    metadata = read_ingest_metadata(find_bronze_run_dir(cfg, run_id))
    try:
        quarantined = (spark.read.format("delta").load(cfg.quarantine_path)
                       .filter(F.col("ingest_run_id") == run_id).count())
    except Exception:
        quarantined = 0
    ledger = {
        "run_id": run_id,
        "rows_in": metadata["rows_in"],
        "rows_quarantined": quarantined,
        "rows_warned": silver.filter(F.col("dq_status") == DQ_STATUS_WARNING).count(),
        "rows_ok": silver.filter(F.col("dq_status") == DQ_STATUS_OK).count(),
    }

summary = {
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "gold_published_at_utc": started.isoformat(),
    **ledger, **gold_counts,
}
(spark.createDataFrame([summary]).write.format("delta")
 .option("mergeSchema", "true").mode("append").save(cfg.runs_path))

log.info(
    "RUN SUMMARY %s | rows_in=%s rows_quarantined=%s rows_warned=%s rows_ok=%s "
    "rows_gold=%d (CA=%d OR=%d WA=%d)",
    summary["run_id"], summary.get("rows_in"), summary.get("rows_quarantined"),
    summary.get("rows_warned"), summary.get("rows_ok"), summary["rows_gold"],
    summary["rows_gold_ca"], summary["rows_gold_or"], summary["rows_gold_wa"],
)
exit_value = json.dumps(summary)
print(exit_value)
mssparkutils.notebook.exit(exit_value)